# Data Audit

**Project question:** What must I check before treating raw retail files as modeling data?

By the end of this notebook, you should be able to:

- separate missing, invalid, duplicated, outlying, and unmatched records
- parse numeric and date fields without hiding conversion failures
- write an auditable issue table before changing the raw data

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [1]:

from lite_setup import ensure_packages
await ensure_packages()

Using the current Python environment.


In [2]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [3]:
sales = pd.read_csv(DATA / 'retail_sales_messy.csv')
stores = pd.read_csv(DATA / 'store_metadata.csv')
print(f'Raw rows: {len(sales)}; columns: {sales.shape[1]}')
sales.head()

Raw rows: 98; columns: 7


,store_id,date,week,units,revenue,price,promotion
0,S001,2026-01-01,1,166.0,1641.74,9.89,0
1,S002,2026-01-01,1,164.0,1656.40,10.10,0
2,S003,2026-01-01,1,181.0,1866.11,10.31,0
3,S004,2026-01-01,1,172.0,1783.64,10.37,0
4,S005,2026-01-01,1,231.0,2312.31,10.01,1


The raw file stays unchanged. We create parsed copies so failed conversions become visible as missing values rather than silently changing the source.

In [4]:
parsed = sales.copy()
for col in ['units', 'revenue', 'price', 'week', 'promotion']:
    parsed[col] = pd.to_numeric(parsed[col], errors='coerce')
parsed['date'] = pd.to_datetime(parsed['date'], errors='coerce')

audit = pd.DataFrame({
    'raw_dtype': sales.dtypes.astype(str),
    'parsed_dtype': parsed.dtypes.astype(str),
    'missing_after_parse': parsed.isna().sum(),
    'unique_nonmissing': parsed.nunique(dropna=True),
})
audit

,raw_dtype,parsed_dtype,missing_after_parse,unique_nonmissing
store_id,str,str,0,9
date,str,datetime64[us],0,12
week,int64,int64,0,12
units,float64,float64,1,65
revenue,float64,float64,1,96
price,float64,float64,0,75
promotion,int64,int64,0,2


Domain rules must be stated, not guessed. For this synthetic retailer, prices should be in dollars between 0 and 50 and weekly units should not exceed 1,000. These cutoffs would require subject-matter confirmation in a real project.

In [5]:
known_stores = set(stores['store_id'])
issues = pd.DataFrame(index=sales.index)
issues['duplicate_record'] = sales.duplicated(keep=False)
issues['missing_required_value'] = parsed[['units', 'revenue', 'price']].isna().any(axis=1)
issues['invalid_date'] = parsed['date'].isna()
issues['nonpositive_price'] = parsed['price'].le(0)
issues['suspected_cents_price'] = parsed['price'].gt(50)
issues['extreme_units'] = parsed['units'].gt(1000)
issues['unmatched_store_key'] = ~sales['store_id'].isin(known_stores)

issue_summary = issues.sum().rename('flagged_rows').to_frame()
issue_summary

,flagged_rows
duplicate_record,2
missing_required_value,2
invalid_date,0
nonpositive_price,1
suspected_cents_price,1
extreme_units,1
unmatched_store_key,1


In [6]:
flagged_rows = pd.concat([sales, issues], axis=1).loc[issues.any(axis=1)]
flagged_rows

,store_id,date,week,units,revenue,price,promotion,duplicate_record,missing_required_value,invalid_date,nonpositive_price,suspected_cents_price,extreme_units,unmatched_store_key
5,S006,2026-01-01,1,NaN,2092.86,10.57,0,False,True,False,False,False,False,False
13,S006,2026-01-08,2,215.0,2324.15,-2.99,0,False,False,False,True,False,False,False
17,S002,2026-01-15,3,170.0,1599.70,9.41,0,True,False,False,False,False,False,False
22,S007,2026-01-15,3,216.0,NaN,11.02,0,False,True,False,False,False,False,False
31,S008,2026-01-22,4,245.0,2557.80,1099.00,0,False,False,False,False,True,False,False
44,S005,2026-02-08,6,1650.0,2388.76,9.79,1,False,False,False,False,False,True,False
96,S002,2026-01-15,3,170.0,1599.70,9.41,0,True,False,False,False,False,False,False
97,S010,2026-02-15,7,205.0,2190.50,10.69,0,False,False,False,False,False,False,True


**Interpretation:** Duplicate flags identify both copies, so the flagged-row total is not the number of rows to remove. A negative price is invalid; a price above $50 is a suspected unit error; an extreme units value is a review item, not automatic proof of an error.

**Transfer exercise:** Write three domain rules for your project data. For each, state whether a violation should be corrected, excluded, or retained with a warning, and identify who can authorize that decision.